## Homework 1: Agentic RAG

This is the personal implementation of the homework 1 of [LLM Zoomcamp 2026 Cohort](https://github.com/DataTalksClub/llm-zoomcamp/tree/main). The homework requires building an agentic RAG for the course lessons. The course repository is organized by modules and each module is a top-level folder with a `lessons/` subfolder of numbered markdown pages:

```python
01-agentic-rag/
└── lessons/
    ├── 01-intro.md
    ├── 02-environment.md
    ├── ...
    └── 16-other-frameworks.md
```

Each lesson page is a single markdown file, which are exactly the data to be fetched from GitHub and to be used as the knowledge base.

In [1]:
# added to make sure updates on helper files appliies at each cell run
%load_ext autoreload
%autoreload 2

In [13]:
# pull the lesson pages from the course repository
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader( # download entire repository and go over the files
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

### Q1. How many lesson pages

In [14]:
# load each file as filename and content pairs
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Total lesson pages: {len(documents)}")

Total lesson pages: 72


In [15]:
# check a sample data
documents[45]

{'content': "# Next Steps\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TlKPBjItUw8&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we covered evaluation at three levels:\n\n1. Search evaluation: Hit Rate and MRR to measure retrieval quality\n2. RAG evaluation: LLM-as-a-judge for answer quality\n3. Agent evaluation: final answers plus tool-call trajectories\n\nEvaluation is not a one-time activity. As you tune search parameters,\nswitch models, or modify prompts, re-run evaluation. Make sure the\nsystem is getting better, not worse.\n\nEvaluation is the most important part of building AI systems. It is also\nthe most time-consuming. Only after evaluation can you be confident\nthat your system works. Validate every change against your evaluation\nframework before going to production. This applies to prompt updates,\nmodel swaps, and agent modifications.\n\n## From synthetic data to real data\n\nThe evaluation workflow in practice:\n\n1. Start with synthetic d

### Q2. Indexing and searching

Documents will be indexed with `minsearch` using `content` as a text field and `filename` as a keyword field. 

Then search will be done with this query: "How does the agentic loop keep calling the model until it stops?"

In [16]:
from minsearch import Index

# define a  function to index with minsearch
def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

In [17]:
# Perform a search with below query
query = "How does the agentic loop keep calling the model until it stops?"

# create the index
index = build_index(documents)

# get all related answers from all the courses (no filter, no boost, no numeric limit)
search_results = index.search(query)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [18]:
print(f"Filename of the first result: {search_results[0]['filename']}")

Filename of the first result: 01-agentic-rag/lessons/14-agentic-loop.md


### Q3. RAG

Build a RAG assistant to answer the query. For this exercise, the original `rag_helper.py` from the course is adapted and renamed as `helper.py`. 

The RAG will reply to the question: "How does the agentic loop keep calling the model until it stops?"

In [19]:
# make sure .env file is loaded
from dotenv import load_dotenv
print(load_dotenv())

# check whether the OpenAI client works
from openai import OpenAI
openai_client = OpenAI()

True


In [20]:
# import RAG class
from helper import RAGBase

# define the RAG
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

# ask to RAG
answer, usage = assistant.rag("How does the agentic loop keep calling the model until it stops?")

print(f"The total number of input tokens sent to the model: {usage.input_tokens}")
print(answer)

The total number of input tokens sent to the model: 7036
The agentic loop keeps calling the model by using a `while True` loop and checking whether the model returned any `function_call` items.

How it works:

1. Call the model with the current `messages`
2. If the response contains a function call:
   - run the tool
   - append the tool result to `messages`
   - set `has_function_calls = True`
3. If there are no function calls in the response, break out of the loop

So the loop stops only when the model returns a response with no tool calls.

If you want, I can also show the loop in plain English or with a simplified pseudocode version.


### Q4. Chunking

To avoid retrieving the whole page for a match, this part applies chunking. Chunking splits each page into smaller overlapping pieces and index those pieces instead.

In [22]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Total number of chunks: {len(chunks)}")

Total number of chunks: 295


### Q5. RAG with chunking

Chunking makes each request smaller since a smaller context is sent to the LLM. This section uses chunks with RAG.

In [23]:
# create the chunks index
chunks_index = build_index(chunks)

# define the RAG
chunked_assistant = RAGBase(
    index=chunks_index,
    llm_client=openai_client,
)

# ask to RAG
chunked_answer, chunked_usage = chunked_assistant.rag("How does the agentic loop keep calling the model until it stops?")

print(f"The total number of input tokens sent to the model: {chunked_usage.input_tokens}")
print(chunked_answer)

The total number of input tokens sent to the model: 2221
It keeps calling the model inside a `while True` loop, and after each call it checks whether any `function_call` items were returned.

- If the model returns a function call, the code runs that tool, appends the result to `messages`, and loops again.
- If the model returns only a final `message` with no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: it repeats until the model stops asking for tools.


In [26]:
print(f"The chunked version sends {int(usage.input_tokens / chunked_usage.input_tokens)} times fewer input tokens!")

The chunked version sends 3 times fewer input tokens!


### Q6. Turning it into an agent

This sections makes the RAG agentic by providing a `search` tool and letting LLM decide when and what to search. For simplicity, this homework is done with the [toyaikit](https://github.com/alexeygrigorev/toyaikit) which was also used during the lectures. 

In [27]:
# import the required classes
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [28]:
# define search with docstring
def chunked_search(query: str) -> dict[str, str]:
    """
    Search the course lessons for entries matching the given query.
    """
    return chunks_index.search(
        query,
        num_results=5
    )

In [29]:
# register it without scheme
agent_tools = Tools()
agent_tools.add_tool(chunked_search)

In [30]:
# check the tool with framework generated scheme
agent_tools.get_tools()

[{'type': 'function',
  'name': 'chunked_search',
  'description': 'Search the course lessons for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [31]:
# define instructions to nudge the agent to search a few times
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

In [32]:
# create the chat interface and a callback, then build the runner
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [33]:
# run a single prompt
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [37]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop plain RAG"}', call_id='call_ykVjrMGI6awzQp140huVOj9O', name='chunked_search', type='function_call', id='fc_07ba0f4781f919fe006a344a412b08819c90d2e1038a6b7d76', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop retrieval augmented generation"}', call_id='call_OCN0PcrLMTnF9DdJibaxTiN0', name='chunked_search', type='function_call', id='fc_07ba0f4781f919fe006a344a412b18819cb433fccd9b0e84ec', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"RAG versus agentic loop"}', call_id='call_DEemcJjB0heij

Answer: The search tool was called 3 times. 